# Title : Analyzing Weather Data from OpenWeatherMap API

In [14]:
import requests
import pandas as pd
import datetime
import numpy as np #for trend calculations
import matplotlib.pyplot as plt
import seaborn as sns
import folium # for geographic vizualization
from getpass import getpass

In [15]:
# Set your OpenWeatherMap API key
api_key = '57879ee26817c8e42515f95a6bdbe38d'

In [16]:
# Set the location for which you want to retrieve weather data
lat = 18.184135
lon = 74.610764

In [17]:
# https://openweathermap.org/api/one-call-3
# how	How to use api call
# Construct the API URL
# api_url = (f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&")
api_url = (
    f"https://api.openweathermap.org/data/2.5/forecast"
    f"?lat={lat}&lon={lon}&appid={api_key}&units=metric"
)
#units=metric to receive temperature in Celsius and wind speed in metres per second

In [18]:

# Send a GET request to the API
response = requests.get(api_url, timeout=30)

# Check API response
if response.status_code == 200:
    weather_data = response.json()
    print("Weather data retrieved successfully!")
else:
    print("Error:", response.status_code)
    print(response.json())
    raise Exception("Unable to retrieve weather data")

# Check response keys
print(weather_data.keys())

# Check number of forecast records
print(len(weather_data['list']))

# Display weather description
print(weather_data['list'][0]['weather'][0]['description'])

# Display location information
print(weather_data['city']['name'])


Error: 401
{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


Exception: Unable to retrieve weather data

In [ ]:

# Extract relevant weather attributes

timestamps = [
    pd.to_datetime(item['dt'], unit='s', utc=True)
    for item in weather_data['list']
]

temperatures = [
    item['main'].get('temp') for item in weather_data['list']
]

humidity = [
    item['main'].get('humidity') for item in weather_data['list']
]

wind_speed = [
    item['wind'].get('speed') for item in weather_data['list']
]

weather_description = [
    item['weather'][0].get('description')
    for item in weather_data['list']
]

# Extract precipitation (rain + snow in mm over 3 hours)
precipitation = [
    item.get('rain', {}).get('3h', 0) +
    item.get('snow', {}).get('3h', 0)
    for item in weather_data['list']
]

# Extract geographical information
latitude = weather_data['city']['coord']['lat']
longitude = weather_data['city']['coord']['lon']


In [ ]:

# Create a Pandas DataFrame

weather_df = pd.DataFrame({
    'Timestamp': timestamps,
    'Temperature': temperatures,
    'Humidity': humidity,
    'Wind_Speed': wind_speed,
    'Precipitation': precipitation,
    'Weather_Description': weather_description
})

# Set Timestamp as DataFrame index
weather_df.set_index('Timestamp', inplace=True)

# Display the DataFrame
print(weather_df.head())


In [ ]:

# Calculate weather statistics

max_temp = weather_df['Temperature'].max()
print(f"Maximum Temperature: {max_temp:.2f} °C")

min_temp = weather_df['Temperature'].min()
print(f"Minimum Temperature: {min_temp:.2f} °C")

avg_temp = weather_df['Temperature'].mean()
print(f"Average Temperature: {avg_temp:.2f} °C")

avg_humidity = weather_df['Humidity'].mean()
print(f"Average Humidity: {avg_humidity:.2f} %")

avg_wind_speed = weather_df['Wind_Speed'].mean()
print(f"Average Wind Speed: {avg_wind_speed:.2f} m/s")

total_precipitation = weather_df['Precipitation'].sum()
print(f"Total Forecast Precipitation: {total_precipitation:.2f} mm")


In [ ]:

# Clean and preprocess the data

# Convert relevant columns to numeric
numeric_columns = [
    'Temperature',
    'Humidity',
    'Wind_Speed',
    'Precipitation'
]

for col in numeric_columns:
    weather_df[col] = pd.to_numeric(
        weather_df[col], errors='coerce'
    )

# Handle missing values
# Interpolate numeric weather observations
weather_df[numeric_columns] = (
    weather_df[numeric_columns]
    .interpolate(method='time', limit_area='inside')
)

# Remove rows with missing essential weather information
weather_df.dropna(
    subset=['Temperature', 'Humidity', 'Wind_Speed'],
    inplace=True
)

# Remove duplicate timestamps
weather_df = weather_df[
    ~weather_df.index.duplicated(keep='first')
]

# Display cleaned data
print(weather_df.head())

# Check missing values
print("\nMissing values:")
print(weather_df.isnull().sum())

# Check data information
print("\nData information:")
print(weather_df.info())

In [ ]:

# Daily aggregation

daily_mean_temp = weather_df['Temperature'].resample('D').mean()
daily_mean_humidity = weather_df['Humidity'].resample('D').mean()
daily_mean_wind_speed = weather_df['Wind_Speed'].resample('D').mean()
daily_total_precipitation = weather_df['Precipitation'].resample('D').sum()

# Daily summary
daily_summary = weather_df.resample('D').agg({
    'Temperature': ['mean', 'max', 'min'],
    'Humidity': 'mean',
    'Wind_Speed': 'mean',
    'Precipitation': 'sum'
})

print("Daily Weather Summary:")
print(daily_summary)

# Monthly aggregation
monthly_summary = weather_df.resample('ME').agg({
    'Temperature': ['mean', 'max', 'min'],
    'Humidity': 'mean',
    'Wind_Speed': 'mean',
    'Precipitation': 'sum'
})

print("\nMonthly Weather Summary:")
print(monthly_summary)

# Seasonal aggregation
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

weather_df['Season'] = weather_df.index.month.map(get_season)

seasonal_summary = weather_df.groupby('Season').agg({
    'Temperature': ['mean', 'max', 'min'],
    'Humidity': 'mean',
    'Wind_Speed': 'mean',
    'Precipitation': 'sum'
})

print("\nSeasonal Weather Summary:")
print(seasonal_summary)


In [ ]:

plt.figure(figsize=(10, 6))

daily_mean_temp.plot(
    color='red',
    linestyle='-',
    marker='o'
)

plt.title('Mean Daily Temperature')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

daily_mean_humidity.plot(
    kind='bar',
    color='blue'
)

plt.title('Mean Daily Humidity')
plt.xlabel('Date')
plt.ylabel('Humidity (%)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

plt.scatter(
    weather_df['Temperature'],
    weather_df['Wind_Speed'],
    color='green'
)

plt.title('Temperature vs Wind Speed')
plt.xlabel('Temperature (°C)')
plt.ylabel('Wind Speed (m/s)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

daily_total_precipitation.plot(
    kind='bar',
    color='skyblue'
)

plt.title('Daily Precipitation')
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(12, 6))

plt.plot(
    weather_df.index,
    weather_df['Temperature'],
    marker='o',
    label='Temperature'
)

plt.plot(
    weather_df.index,
    weather_df['Temperature'].rolling(
        window=3, min_periods=1
    ).mean(),
    color='red',
    linewidth=2,
    label='3-interval Rolling Average'
)

plt.title('Temperature Trend Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Calculate correlation matrix

correlation_matrix = weather_df[
    ['Temperature', 'Humidity', 'Wind_Speed', 'Precipitation']
].corr()

# Plot correlation heatmap

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Weather Attributes Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

plt.scatter(
    weather_df['Temperature'],
    weather_df['Humidity'],
    color='purple'
)

plt.xlabel('Temperature (°C)')
plt.ylabel('Humidity (%)')
plt.title('Temperature vs Humidity Scatter Plot')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Create a geographical map

weather_map = folium.Map(
    location=[latitude, longitude],
    zoom_start=10
)

# Add weather location marker

folium.Marker(
    location=[latitude, longitude],
    popup=f"""
    Location: {weather_data['city']['name']}<br>
    Latitude: {latitude}<br>
    Longitude: {longitude}<br>
    Average Temperature: {avg_temp:.2f} °C<br>
    Average Humidity: {avg_humidity:.2f} %
    """,
    tooltip="Weather Location",
    icon=folium.Icon(color='red', icon='cloud')
).add_to(weather_map)

# Display map
weather_map
